# 04 — Phân lớp có giám sát (họ bài toán phụ) · Đồ án 6: Phát hiện giao dịch thẻ gian lận
**Phần E (Mong).** Câu hỏi nghiệp vụ: *với ngân sách rà soát 100 giao dịch mỗi ngày, mô hình phân lớp có giám sát bắt được bao nhiêu giao dịch gian lận so với baseline?*

**Trạng thái giao thức đánh giá: ĐỀ XUẤT, chưa chốt với Huy (D) và Cường (B).** Nếu giao thức đổi, chỉ sửa ô cấu hình ở §1 và ô chia dữ liệu ở §2.

Giao thức đề xuất:
1. Test = ngày 1 (`Time >= 86400`), chỉ chạy **một lần** ở §7.
2. Train + validation = ngày 0; validation là 30% **cuối** của ngày 0 theo thời gian.
3. Đặc trưng: `V1..V28` và `Amount` (log1p). Bỏ `Time` thô.
4. Chỉ số: Precision@100, Recall@100, PR-AUC (Average Precision). Không dùng accuracy.
5. Baseline: đoán lớp đa số và xếp hạng ngẫu nhiên.

## 1. Cấu hình và nạp dữ liệu

In [1]:
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, accuracy_score, confusion_matrix

RANDOM_STATE = 42
K = 100                      # ngân sách rà soát mỗi ngày (áp dụng cho test = 1 ngày)
LABEL = "Class"
DAY_SECONDS = 86400
VAL_FRACTION = 0.30          # phần cuối của ngày 0 làm validation
K_VAL = round(K * VAL_FRACTION)   # validation chỉ là 30% một ngày nên ngân sách tỉ lệ: 30
DATA = Path("../data/raw/creditcard.csv")
OUT = Path("../outputs")
EXPECTED_SHA256 = "76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89"
CHECK_SHA256 = True          # đặt False nếu chạy thử trên dữ liệu giả

if CHECK_SHA256:
    sha = hashlib.sha256(DATA.read_bytes()).hexdigest()
    assert sha == EXPECTED_SHA256, f"sha256 khác bản của nhóm: {sha}"
df = pd.read_csv(DATA)
print("shape:", df.shape, "| fraud:", int(df[LABEL].sum()))

shape: (284807, 31) | fraud: 492


## 2. Chia dữ liệu theo thời gian
Ngày 0 và ngày 1 lấy từ `Time // 86400`. Theo audit của Khang (và đã kiểm lại trên file thật): ngày 0 có 144.786 dòng, 281 gian lận; ngày 1 có 140.021 dòng, 211 gian lận; không dòng nào ở ngày 1 trùng với ngày 0.

In [2]:
df = df.sort_values("Time", kind="stable").reset_index().rename(columns={"index": "row_id"})
df["rel_day"] = (df["Time"] // DAY_SECONDS).astype(int)
FEATURES = [f"V{i}" for i in range(1, 29)] + ["Amount"]

day0 = df[df.rel_day == 0]
test = df[df.rel_day == 1]
cut = int(len(day0) * (1 - VAL_FRACTION))
train, val = day0.iloc[:cut], day0.iloc[cut:]

for name, part in [("train", train), ("val", val), ("test (ngày 1)", test)]:
    print(f"{name:14s} n={len(part):>7d}  fraud={int(part[LABEL].sum()):>4d}")

# kiểm tra rò rỉ: dòng test trùng hoàn toàn với dòng ngày 0
cols = FEATURES + [LABEL]
dup = test[cols].merge(day0[cols].drop_duplicates(), on=cols, how="inner")
print("dòng test trùng với ngày 0:", len(dup))
assert len(dup) == 0, "có dòng trùng giữa test và train: cần xử lý với Cường"

n_fraud_test = int(test[LABEL].sum())
print(f"trần lý thuyết Recall@{K} ở ngày 1 = {min(K, n_fraud_test) / n_fraud_test:.3f}")

train          n= 101350  fraud= 224
val            n=  43436  fraud=  57
test (ngày 1)  n= 140021  fraud= 211
dòng test trùng với ngày 0: 0
trần lý thuyết Recall@100 ở ngày 1 = 0.474


## 3. Hàm đánh giá

In [3]:
def precision_at_k(y, score, k=K):
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(np.asarray(y)[idx].mean())

def recall_at_k(y, score, k=K):
    y = np.asarray(y)
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(y[idx].sum() / y.sum())

def report(name, y, score, k=K):
    return {"model": name,
            "PR_AUC": average_precision_score(y, score),
            f"Precision@{k}": precision_at_k(y, score, k),
            f"Recall@{k}": recall_at_k(y, score, k)}

## 4. Baseline
- **Đa số:** luôn đoán không gian lận. Accuracy rất cao nhưng bắt được 0 giao dịch gian lận, nên accuracy không được dùng làm chỉ số chính.
- **Xếp hạng ngẫu nhiên:** 100 lần chạy, lấy trung bình.

In [4]:
yv, yt = val[LABEL].values, test[LABEL].values
rows = []

maj = np.zeros(len(yt), dtype=int)
print(f"Đa số: accuracy trên test = {accuracy_score(yt, maj):.4f}, số gian lận bắt được = 0")

rng = np.random.default_rng(RANDOM_STATE)
def random_baseline(y, k, runs=100):
    res = [report("random", y, rng.random(len(y)), k) for _ in range(runs)]
    return pd.DataFrame(res).drop(columns="model").mean().to_dict()

rb_val, rb_test = random_baseline(yv, K_VAL), random_baseline(yt, K)
print("Random (validation):", {k: round(v, 5) for k, v in rb_val.items()})
print("Random (test):      ", {k: round(v, 5) for k, v in rb_test.items()})

Đa số: accuracy trên test = 0.9985, số gian lận bắt được = 0
Random (validation): {'PR_AUC': 0.00158, 'Precision@30': 0.00067, 'Recall@30': 0.00035}
Random (test):       {'PR_AUC': 0.00156, 'Precision@100': 0.0015, 'Recall@100': 0.00071}


## 5. Mô hình trên validation
Ba mô hình, mỗi mô hình một vài cấu hình nhỏ. Xử lý mất cân bằng bằng trọng số lớp, chỉ học trên train. Validation chỉ là 30% của một ngày nên ngân sách rà soát tỉ lệ theo (`K_VAL` = 30), và chỉ có số ít giao dịch gian lận nên kết quả dao động; không nên diễn giải sự chênh lệch nhỏ giữa các mô hình. Cấu hình được chọn theo PR-AUC, ít nhạy hơn với K.

In [5]:
amount_log = FunctionTransformer(np.log1p, feature_names_out="one-to-one")
v_cols = [c for c in FEATURES if c != "Amount"]

def prep():
    return ColumnTransformer(
        [("amount", Pipeline([("log", amount_log), ("sc", StandardScaler())]), ["Amount"])],
        remainder="passthrough")   # V1..V28 đã là thành phần PCA, giữ nguyên

def make(name, **kw):
    if name == "logreg":
        clf = LogisticRegression(class_weight="balanced", max_iter=2000,
                                 random_state=RANDOM_STATE, **kw)
    elif name == "random_forest":
        clf = RandomForestClassifier(class_weight="balanced_subsample", n_jobs=-1,
                                     random_state=RANDOM_STATE, **kw)
    else:
        clf = HistGradientBoostingClassifier(class_weight="balanced",
                                             random_state=RANDOM_STATE, **kw)
    return Pipeline([("prep", prep()), ("clf", clf)])

grid = {
    "logreg": [{"C": 0.1}, {"C": 1.0}],
    "random_forest": [{"n_estimators": 200, "min_samples_leaf": 1},
                      {"n_estimators": 200, "min_samples_leaf": 5}],
    "hist_gb": [{"max_depth": 4, "learning_rate": 0.1},
                {"max_depth": 6, "learning_rate": 0.05}],
}

X_tr, y_tr, X_val, y_val = train[FEATURES], train[LABEL], val[FEATURES], val[LABEL]
val_rows, fitted = [], {}
for name, cfgs in grid.items():
    for cfg in cfgs:
        m = make(name, **cfg).fit(X_tr, y_tr)
        s = m.predict_proba(X_val)[:, 1]
        r = report(name, y_val, s, K_VAL); r["config"] = json.dumps(cfg)
        val_rows.append(r)
val_table = pd.DataFrame(val_rows).sort_values("PR_AUC", ascending=False)
val_table

,model,PR_AUC,Precision@30,Recall@30,config
3,random_forest,0.779151,0.933333,0.491228,"{""n_estimators"": 200, ""min_samples_leaf"": 5}"
2,random_forest,0.765992,0.933333,0.491228,"{""n_estimators"": 200, ""min_samples_leaf"": 1}"
0,logreg,0.761632,0.900000,0.473684,"{""C"": 0.1}"
5,hist_gb,0.760507,0.966667,0.508772,"{""max_depth"": 6, ""learning_rate"": 0.05}"
1,logreg,0.750943,0.933333,0.491228,"{""C"": 1.0}"
4,hist_gb,0.750746,0.966667,0.508772,"{""max_depth"": 4, ""learning_rate"": 0.1}"


## 6. Chọn cấu hình
Chọn cấu hình có PR-AUC cao nhất trên validation cho **mỗi** họ mô hình, để bảng cuối so sánh các mô hình mà không dùng test để chọn tham số.

In [6]:
best = (val_table.sort_values("PR_AUC", ascending=False)
        .groupby("model", as_index=False).head(1).reset_index(drop=True))
best

,model,PR_AUC,Precision@30,Recall@30,config
0,random_forest,0.779151,0.933333,0.491228,"{""n_estimators"": 200, ""min_samples_leaf"": 5}"
1,logreg,0.761632,0.900000,0.473684,"{""C"": 0.1}"
2,hist_gb,0.760507,0.966667,0.508772,"{""max_depth"": 6, ""learning_rate"": 0.05}"


## 7. Đánh giá cuối trên test (ngày 1) — chạy một lần
Huấn luyện lại trên toàn bộ ngày 0 (train + validation) với cấu hình đã chọn, rồi chấm điểm ngày 1.

In [7]:
X_d0, y_d0 = day0[FEATURES], day0[LABEL]
X_te, y_te = test[FEATURES], test[LABEL]

final_rows, test_scores = [], {}
for _, b in best.iterrows():
    m = make(b["model"], **json.loads(b["config"])).fit(X_d0, y_d0)
    s = m.predict_proba(X_te)[:, 1]
    test_scores[b["model"]] = s
    final_rows.append(report(b["model"], y_te, s))
final_rows.append({"model": "random_baseline_mean", **rb_test})
final = pd.DataFrame(final_rows)
final

,model,PR_AUC,Precision@100,Recall@100
0,random_forest,0.797969,0.9300,0.440758
1,logreg,0.754060,0.9600,0.454976
2,hist_gb,0.700876,0.9500,0.450237
3,random_baseline_mean,0.001558,0.0015,0.000711


## 8. Phân tích lỗi (mô hình tốt nhất theo PR-AUC trên test)

In [8]:
top = final[final.model != "random_baseline_mean"].sort_values("PR_AUC", ascending=False).iloc[0]["model"]
s = test_scores[top]
order = np.argsort(-s, kind="stable")[:K]
flagged = test.iloc[order]
tp = int(flagged[LABEL].sum()); fp = K - tp
missed = test[(test[LABEL] == 1) & (~test.index.isin(flagged.index))]
print(f"Mô hình: {top} | trong top {K}: đúng {tp}, sai {fp} | bỏ sót {len(missed)}/{n_fraud_test}")
print("Amount của giao dịch gian lận bị bỏ sót:")
print(missed["Amount"].describe().round(2))

Mô hình: random_forest | trong top 100: đúng 93, sai 7 | bỏ sót 118/211
Amount của giao dịch gian lận bị bỏ sót:
count     118.00
mean      149.96
std       339.21
min         0.00
25%         1.00
50%         7.12
75%        90.24
max      2125.87
Name: Amount, dtype: float64


## 9. Lưu kết quả cho dashboard và so sánh với Huy
`classification_scores_test.csv` có `row_id` (số thứ tự dòng trong `creditcard.csv` gốc) để nối với `anomaly_scores_test.csv` của Huy. Quỳnh dùng file này để nạp vào bảng fact khi nhóm chốt schema.

In [9]:
OUT.mkdir(exist_ok=True)
final.to_csv(OUT / "classification_results.csv", index=False)
val_table.to_csv(OUT / "classification_validation.csv", index=False)
scores = test[["row_id", "Time", "Amount", LABEL]].copy()
for k, v in test_scores.items():
    scores[f"score_{k}"] = v
scores.to_csv(OUT / "classification_scores_test.csv", index=False)
print("đã lưu:", [p.name for p in OUT.glob("classification_*")])

đã lưu: ['classification_results.csv', 'classification_scores_test.csv', 'classification_validation.csv']


## 10. Hạn chế và ghi chú
- Test chỉ có 211 giao dịch gian lận trong ngày 1; một vài giao dịch thay đổi kết quả đáng kể. Không kết luận hơn kém giữa các mô hình nếu chênh lệch nhỏ.
- V1..V28 là thành phần PCA đã ẩn danh nên không diễn giải ý nghĩa nghiệp vụ của từng cột.
- Mô hình này học từ nhãn; anomaly detection (Huy) thì không. Trong thực tế không phải lúc nào cũng có nhãn.
- Giao thức chia dữ liệu và xử lý dòng trùng cần được Huy và Cường xác nhận trước khi dùng số liệu trong báo cáo.
- Khai báo sử dụng AI: ghi theo mẫu của nhóm (công cụ, mục đích, phần được chỉnh sửa).